In [2]:
!pip install wandb sacrebleu gdown datasets spacy -q
!python -m spacy download en_core_web_sm -q
!python -m spacy download de_core_news_sm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 1.4 MB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 62.1 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 32.6 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: da25m017 (da25m017-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
%%writefile dataset.py
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import spacy
from collections import Counter

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"

PAD_IDX = 0
UNK_IDX = 1
SOS_IDX = 2
EOS_IDX = 3

class Multi30kDataset(Dataset):
    def __init__(self, split='train'):
        self.split = split
        raw = load_dataset("bentrevett/multi30k")
        split_map = {'train': 'train', 'val': 'validation', 'test': 'test'}
        self.data = raw[split_map[split]]
        self.de_nlp = spacy.load("de_core_news_sm")
        self.en_nlp = spacy.load("en_core_web_sm")
        self.src_vocab = None
        self.tgt_vocab = None
        self.src_itos  = None
        self.tgt_itos  = None
        self.processed = None

    def tokenize_de(self, text):
        return [tok.text.lower() for tok in self.de_nlp.tokenizer(text)]

    def tokenize_en(self, text):
        return [tok.text.lower() for tok in self.en_nlp.tokenizer(text)]

    def build_vocab(self, de_sentences=None, en_sentences=None, min_freq=1):
        if de_sentences is None:
            de_sentences = [self.tokenize_de(item["de"]) for item in self.data]
        if en_sentences is None:
            en_sentences = [self.tokenize_en(item["en"]) for item in self.data]
        def make_vocab(tokenized):
            counter = Counter()
            for tokens in tokenized:
                counter.update(tokens)
            vocab = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX,
                     SOS_TOKEN: SOS_IDX, EOS_TOKEN: EOS_IDX}
            for word, freq in counter.items():
                if freq >= min_freq and word not in vocab:
                    vocab[word] = len(vocab)
            itos = {idx: tok for tok, idx in vocab.items()}
            return vocab, itos
        self.src_vocab, self.src_itos = make_vocab(de_sentences)
        self.tgt_vocab, self.tgt_itos = make_vocab(en_sentences)

    def set_vocab(self, src_vocab, src_itos, tgt_vocab, tgt_itos):
        self.src_vocab = src_vocab
        self.src_itos  = src_itos
        self.tgt_vocab = tgt_vocab
        self.tgt_itos  = tgt_itos

    def process_data(self):
        assert self.src_vocab is not None
        self.processed = []
        for item in self.data:
            de_tokens = self.tokenize_de(item["de"])
            en_tokens = self.tokenize_en(item["en"])
            src_ids = [SOS_IDX] + [self.src_vocab.get(t, UNK_IDX) for t in de_tokens] + [EOS_IDX]
            tgt_ids = [SOS_IDX] + [self.tgt_vocab.get(t, UNK_IDX) for t in en_tokens] + [EOS_IDX]
            self.processed.append((
                torch.tensor(src_ids, dtype=torch.long),
                torch.tensor(tgt_ids, dtype=torch.long)
            ))

    def __len__(self):
        return len(self.processed)

    def __getitem__(self, idx):
        return self.processed[idx]

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_padded = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_IDX)
    return src_padded, tgt_padded

def get_dataloaders(batch_size=64):
    print("Loading train split...")
    train_ds = Multi30kDataset(split='train')
    train_ds.build_vocab()
    train_ds.process_data()
    print("Loading val split...")
    val_ds = Multi30kDataset(split='val')
    val_ds.set_vocab(train_ds.src_vocab, train_ds.src_itos,
                     train_ds.tgt_vocab, train_ds.tgt_itos)
    val_ds.process_data()
    print("Loading test split...")
    test_ds = Multi30kDataset(split='test')
    test_ds.set_vocab(train_ds.src_vocab, train_ds.src_itos,
                      train_ds.tgt_vocab, train_ds.tgt_itos)
    test_ds.process_data()
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds,  batch_size=1,          shuffle=False, collate_fn=collate_fn)
    vocab_info = {
        "src_vocab": train_ds.src_vocab, "src_itos": train_ds.src_itos,
        "tgt_vocab": train_ds.tgt_vocab, "tgt_itos": train_ds.tgt_itos,
    }
    return train_loader, val_loader, test_loader, vocab_info

Writing dataset.py


In [6]:
%%writefile lr_scheduler.py
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import LRScheduler

class NoamScheduler(LRScheduler):
    def __init__(self, optimizer, d_model, warmup_steps, last_epoch=-1):
        self.d_model      = d_model
        self.warmup_steps = warmup_steps
        super().__init__(optimizer, last_epoch)

    def _get_lr_scale(self):
        step = self.last_epoch + 1
        return (self.d_model ** -0.5) * min(step ** -0.5, step * self.warmup_steps ** -1.5)

    def get_lr(self):
        scale = self._get_lr_scale()
        return [base_lr * scale for base_lr in self.base_lrs]

Writing lr_scheduler.py


In [10]:
%%writefile model.py
import math, copy, os, gdown, spacy
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataset import Multi30kDataset, PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    attn_w = torch.nan_to_num(F.softmax(scores, dim=-1), nan=0.0)
    return torch.matmul(attn_w, V), attn_w

def make_src_mask(src, pad_idx=PAD_IDX):
    return (src == pad_idx).unsqueeze(1).unsqueeze(2)

def make_tgt_mask(tgt, pad_idx=PAD_IDX):
    tgt_len  = tgt.size(1)
    pad_mask = (tgt == pad_idx).unsqueeze(1).unsqueeze(2)
    causal   = torch.triu(torch.ones(tgt_len, tgt_len, device=tgt.device), diagonal=1).bool()
    return pad_mask | causal

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model; self.num_heads = num_heads; self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(p=dropout)

    def split_heads(self, x):
        B, S, _ = x.size()
        return x.view(B, S, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))
        out, _ = scaled_dot_product_attention(Q, K, V, mask)
        return self.W_o(out.transpose(1, 2).contiguous().view(B, -1, self.d_model))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout  = nn.Dropout(p=dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn   = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, src_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, src_mask)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn   = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, memory, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, memory, memory, src_mask)))
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x

class Encoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
        self.norm   = nn.LayerNorm(layer.norm1.normalized_shape[0])

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class Decoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
        self.norm   = nn.LayerNorm(layer.norm1.normalized_shape[0])

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

class Transformer(nn.Module):
    def __init__(self, src_vocab_size=18669, tgt_vocab_size=9797,
                 d_model=256, N=3, num_heads=8, d_ff=512, dropout=0.1,
                 max_len=256, checkpoint_path="best_model.pt",
                 gdrive_id="1CtH2Ac29z0yifIay04HKuF_9BV4k6VpW"):
        super().__init__()
        self.d_model = d_model
        self.src_vocab_size = src_vocab_size
        self.tgt_vocab_size = tgt_vocab_size
        self.src_embed = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_IDX)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_enc   = PositionalEncoding(d_model, dropout, max_len)
        enc_layer = EncoderLayer(d_model, num_heads, d_ff, dropout)
        dec_layer = DecoderLayer(d_model, num_heads, d_ff, dropout)
        self.encoder     = Encoder(enc_layer, N)
        self.decoder     = Decoder(dec_layer, N)
        self.output_proj = nn.Linear(d_model, tgt_vocab_size)
        self._init_weights()
        self._load_vocab_and_tokenizers()

        # only attempt download/load if checkpoint_path is actually specified
        if checkpoint_path is not None:
            if gdrive_id is not None and not os.path.exists(checkpoint_path):
                gdown.download(id=gdrive_id, output=checkpoint_path, quiet=False)
            if os.path.exists(checkpoint_path):
                ckpt = torch.load(checkpoint_path, map_location='cpu')
                self.load_state_dict(ckpt['model_state_dict'])
                print(f"Loaded weights from {checkpoint_path}")

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _load_vocab_and_tokenizers(self):
        self.de_nlp = spacy.load("de_core_news_sm")
        self.en_nlp = spacy.load("en_core_web_sm")
        train_ds = Multi30kDataset(split='train')
        train_ds.build_vocab()
        self.src_vocab = train_ds.src_vocab
        self.tgt_vocab = train_ds.tgt_vocab
        self.tgt_itos  = train_ds.tgt_itos

    def encode(self, src, src_mask):
        return self.encoder(self.pos_enc(self.src_embed(src) * math.sqrt(self.d_model)), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.output_proj(self.decoder(
            self.pos_enc(self.tgt_embed(tgt) * math.sqrt(self.d_model)),
            memory, src_mask, tgt_mask))

    def forward(self, src, tgt, src_mask, tgt_mask):
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

    def infer(self, src_sentence: str) -> str:
        self.eval()
        device = next(self.parameters()).device
        tokens  = [tok.text.lower() for tok in self.de_nlp.tokenizer(src_sentence)]
        src_ids = [SOS_IDX] + [self.src_vocab.get(t, UNK_IDX) for t in tokens] + [EOS_IDX]
        src     = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)
        with torch.no_grad():
            memory = self.encode(src, make_src_mask(src).to(device))
            ys = torch.tensor([[SOS_IDX]], dtype=torch.long, device=device)
            for _ in range(100):
                tgt_mask = make_tgt_mask(ys).to(device)
                out      = self.decode(memory, make_src_mask(src).to(device), ys, tgt_mask)
                next_tok = out[:, -1, :].argmax(dim=-1).item()
                ys = torch.cat([ys, torch.tensor([[next_tok]], device=device)], dim=1)
                if next_tok == EOS_IDX:
                    break
        special = {PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX}
        return " ".join(self.tgt_itos[i] for i in ys[0].tolist() if i not in special)

Overwriting model.py


In [11]:
import torch, torch.nn as nn, torch.nn.functional as F, math, sacrebleu
from model import Transformer, make_src_mask, make_tgt_mask
from dataset import get_dataloaders, PAD_IDX, SOS_IDX, EOS_IDX
from lr_scheduler import NoamScheduler
import wandb

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size, pad_idx, smoothing=0.1):
        super().__init__()
        self.vocab_size = vocab_size; self.pad_idx = pad_idx
        self.smoothing  = smoothing;  self.confidence = 1.0 - smoothing
    def forward(self, logits, target):
        log_probs  = F.log_softmax(logits, dim=-1)
        smooth_val = self.smoothing / (self.vocab_size - 2)
        with torch.no_grad():
            dist = torch.full_like(log_probs, smooth_val)
            dist[:, self.pad_idx] = 0.0
            dist.scatter_(1, target.unsqueeze(1), self.confidence)
        pad_mask = (target == self.pad_idx)
        loss = -(dist * log_probs).sum(dim=-1).masked_fill(pad_mask, 0.0)
        return loss.sum() / (~pad_mask).sum().clamp(min=1)

def greedy_decode(model, src, src_mask, max_len, start_symbol, end_symbol, device):
    model.eval()
    with torch.no_grad():
        memory = model.encode(src, src_mask)
        ys = torch.tensor([[start_symbol]], dtype=torch.long, device=device)
        for _ in range(max_len - 1):
            tgt_mask = make_tgt_mask(ys).to(device)
            out      = model.decode(memory, src_mask, ys, tgt_mask)
            next_tok = out[:, -1, :].argmax(dim=-1).item()
            ys = torch.cat([ys, torch.tensor([[next_tok]], dtype=torch.long, device=device)], dim=1)
            if next_tok == end_symbol:
                break
    return ys

def evaluate_bleu(model, dataloader, tgt_itos, device, max_len=100):
    model.eval()
    hyps, refs = [], []
    special = {PAD_IDX, SOS_IDX, EOS_IDX}
    with torch.no_grad():
        for src, tgt in dataloader:
            for i in range(src.size(0)):
                s  = src[i].unsqueeze(0).to(device)
                sm = make_src_mask(s).to(device)
                pred = greedy_decode(model, s, sm, max_len, SOS_IDX, EOS_IDX, device)
                hyps.append(" ".join(tgt_itos[j] for j in pred[0].tolist() if j not in special))
                refs.append(" ".join(tgt_itos[j] for j in tgt[i].tolist()  if j not in special))
    return sacrebleu.corpus_bleu(hyps, [refs], force=True).score

def run_epoch(loader, model, loss_fn, optimizer, scheduler, is_train, device):
    model.train() if is_train else model.eval()
    total_loss, total_tok = 0.0, 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for src, tgt in loader:
            src = src.to(device); tgt = tgt.to(device)
            tgt_in = tgt[:, :-1]; tgt_out = tgt[:, 1:]
            sm = make_src_mask(src).to(device)
            tm = make_tgt_mask(tgt_in).to(device)
            logits  = model(src, tgt_in, sm, tm)
            B, T, V = logits.shape
            loss    = loss_fn(logits.reshape(B*T, V), tgt_out.reshape(B*T))
            if is_train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                if scheduler: scheduler.step()
            non_pad      = (tgt_out != PAD_IDX).sum().item()
            total_loss  += loss.item() * non_pad
            total_tok   += non_pad
    return total_loss / max(total_tok, 1)

def build_model(src_vocab_size, tgt_vocab_size):
    return Transformer(src_vocab_size=src_vocab_size, tgt_vocab_size=tgt_vocab_size,
                       d_model=256, N=3, num_heads=8, d_ff=512, dropout=0.1,
                       checkpoint_path=None).to(DEVICE)

print("Loading data...")
train_loader, val_loader, _, vocab_info = get_dataloaders(batch_size=128)
src_vocab_size = len(vocab_info["src_vocab"])
tgt_vocab_size = len(vocab_info["tgt_vocab"])
tgt_itos       = vocab_info["tgt_itos"]
print("Ready.")

Device: cuda
Loading data...
Loading train split...
Loading val split...
Loading test split...
Ready.


In [11]:
%%writefile train.py
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import wandb
import sacrebleu

from model import Transformer, make_src_mask, make_tgt_mask
from dataset import get_dataloaders, PAD_IDX, SOS_IDX, EOS_IDX
from lr_scheduler import NoamScheduler


class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size, pad_idx, smoothing=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_idx    = pad_idx
        self.smoothing  = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, logits, target):
        log_probs  = F.log_softmax(logits, dim=-1)
        smooth_val = self.smoothing / (self.vocab_size - 2)
        with torch.no_grad():
            dist = torch.full_like(log_probs, smooth_val)
            dist[:, self.pad_idx] = 0.0
            dist.scatter_(1, target.unsqueeze(1), self.confidence)
        pad_mask = (target == self.pad_idx)
        loss = -(dist * log_probs).sum(dim=-1)
        loss = loss.masked_fill(pad_mask, 0.0)
        return loss.sum() / (~pad_mask).sum().clamp(min=1)


def run_epoch(data_iter, model, loss_fn, optimizer, scheduler=None,
              epoch_num=0, is_train=True, device="cpu"):
    model.train() if is_train else model.eval()
    total_loss, total_tokens = 0.0, 0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for src, tgt in data_iter:
            src     = src.to(device)
            tgt     = tgt.to(device)
            tgt_in  = tgt[:, :-1]
            tgt_out = tgt[:, 1:]
            src_mask = make_src_mask(src).to(device)
            tgt_mask = make_tgt_mask(tgt_in).to(device)
            logits   = model(src, tgt_in, src_mask, tgt_mask)
            B, T, V  = logits.shape
            loss     = loss_fn(logits.reshape(B*T, V), tgt_out.reshape(B*T))

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()

            non_pad       = (tgt_out != PAD_IDX).sum().item()
            total_loss   += loss.item() * non_pad
            total_tokens += non_pad

    return total_loss / max(total_tokens, 1)


def greedy_decode(model, src, src_mask, max_len, start_symbol, end_symbol, device="cpu"):
    model.eval()
    with torch.no_grad():
        memory = model.encode(src, src_mask)
        ys = torch.tensor([[start_symbol]], dtype=torch.long, device=device)
        for _ in range(max_len - 1):
            tgt_mask = make_tgt_mask(ys).to(device)
            out      = model.decode(memory, src_mask, ys, tgt_mask)
            next_tok = out[:, -1, :].argmax(dim=-1).item()
            ys = torch.cat([ys, torch.tensor([[next_tok]], dtype=torch.long, device=device)], dim=1)
            if next_tok == end_symbol:
                break
    return ys


def evaluate_bleu(model, test_dataloader, tgt_itos, device="cpu", max_len=100):
    model.eval()
    hypotheses, references = [], []
    special = {PAD_IDX, SOS_IDX, EOS_IDX}

    with torch.no_grad():
        for src, tgt in test_dataloader:
            for i in range(src.size(0)):
                src_i    = src[i].unsqueeze(0).to(device)
                src_mask = make_src_mask(src_i).to(device)
                pred_ids = greedy_decode(model, src_i, src_mask, max_len,
                                         SOS_IDX, EOS_IDX, device)
                pred = " ".join(tgt_itos[j] for j in pred_ids[0].tolist() if j not in special)
                ref  = " ".join(tgt_itos[j] for j in tgt[i].tolist()      if j not in special)
                hypotheses.append(pred)
                references.append(ref)

    return sacrebleu.corpus_bleu(hypotheses, [references]).score


def save_checkpoint(model, optimizer, scheduler, epoch, path="best_model.pt"):
    torch.save({
        "epoch":                epoch,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "model_config": {
            "src_vocab_size": model.src_vocab_size,
            "tgt_vocab_size": model.tgt_vocab_size,
            "d_model":   model.d_model,
            "N":         len(model.encoder.layers),
            "num_heads": model.encoder.layers[0].self_attn.num_heads,
            "d_ff":      model.encoder.layers[0].ffn.linear1.out_features,
            "dropout":   0.1,
        }
    }, path)


def load_checkpoint(path, model, optimizer=None, scheduler=None):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    return ckpt["epoch"]


def run_training_experiment(
    d_model      = 256,
    N            = 3,
    num_heads    = 8,
    d_ff         = 512,
    dropout      = 0.1,
    batch_size   = 128,
    num_epochs   = 20,
    warmup_steps = 4000,
    smoothing    = 0.1,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    wandb.init(project="da6401-a3", name="main_training", config={
        "d_model": d_model, "N": N, "num_heads": num_heads,
        "d_ff": d_ff, "dropout": dropout, "batch_size": batch_size,
        "num_epochs": num_epochs, "warmup_steps": warmup_steps,
        "smoothing": smoothing,
    })

    train_loader, val_loader, test_loader, vocab_info = get_dataloaders(batch_size)
    src_vocab_size = len(vocab_info["src_vocab"])
    tgt_vocab_size = len(vocab_info["tgt_vocab"])
    tgt_itos       = vocab_info["tgt_itos"]

    model = Transformer(
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        d_model=d_model, N=N, num_heads=num_heads,
        d_ff=d_ff, dropout=dropout,
        checkpoint_path=None,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
    scheduler = NoamScheduler(optimizer, d_model=d_model, warmup_steps=warmup_steps)
    loss_fn   = LabelSmoothingLoss(tgt_vocab_size, PAD_IDX, smoothing)

    best_val_bleu = 0.0

    for epoch in range(num_epochs):
        train_loss = run_epoch(train_loader, model, loss_fn, optimizer, scheduler,
                               epoch_num=epoch, is_train=True, device=device)
        val_loss   = run_epoch(val_loader, model, loss_fn, None, None,
                               epoch_num=epoch, is_train=False, device=device)
        val_bleu   = evaluate_bleu(model, val_loader, tgt_itos, device)

        print(f"Epoch {epoch+1:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_bleu={val_bleu:.2f}")
        wandb.log({"epoch": epoch+1, "train_loss": train_loss,
                   "val_loss": val_loss, "val_bleu": val_bleu,
                   "lr": optimizer.param_groups[0]["lr"]})

        if val_bleu > best_val_bleu:
            best_val_bleu = val_bleu
            save_checkpoint(model, optimizer, scheduler, epoch, "best_model.pt")
            print(f"  -> Saved best checkpoint (val_bleu={val_bleu:.2f})")

    load_checkpoint("best_model.pt", model)
    test_bleu = evaluate_bleu(model, test_loader, tgt_itos, device)
    print(f"\nTest BLEU: {test_bleu:.2f}")
    wandb.log({"test_bleu": test_bleu})
    wandb.finish()


if __name__ == "__main__":
    run_training_experiment()

Overwriting train.py


In [12]:
from train import run_training_experiment
run_training_experiment()

Using device: cuda


Loading train split...
Loading val split...
Loading test split...


RuntimeError: The size of tensor a (128) must match the size of tensor b (16384) at non-singleton dimension 1

In [13]:
!head -80 /kaggle/working/train.py | grep -n "for i in range"

In [14]:
import torch, sacrebleu, math
import torch.nn as nn
import torch.nn.functional as F
from dataset import get_dataloaders, PAD_IDX, SOS_IDX, EOS_IDX
from model import Transformer, make_src_mask, make_tgt_mask
from lr_scheduler import NoamScheduler


class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size, pad_idx, smoothing=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_idx    = pad_idx
        self.smoothing  = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, logits, target):
        log_probs  = F.log_softmax(logits, dim=-1)
        smooth_val = self.smoothing / (self.vocab_size - 2)
        with torch.no_grad():
            dist = torch.full_like(log_probs, smooth_val)
            dist[:, self.pad_idx] = 0.0
            dist.scatter_(1, target.unsqueeze(1), self.confidence)
        pad_mask = (target == self.pad_idx)
        loss = -(dist * log_probs).sum(dim=-1)
        loss = loss.masked_fill(pad_mask, 0.0)
        return loss.sum() / (~pad_mask).sum().clamp(min=1)


def greedy_decode(model, src, src_mask, max_len, start_symbol, end_symbol, device):
    model.eval()
    with torch.no_grad():
        memory = model.encode(src, src_mask)
        ys = torch.tensor([[start_symbol]], dtype=torch.long, device=device)
        for _ in range(max_len - 1):
            tgt_mask = make_tgt_mask(ys).to(device)
            out      = model.decode(memory, src_mask, ys, tgt_mask)
            next_tok = out[:, -1, :].argmax(dim=-1).item()
            ys = torch.cat([ys, torch.tensor([[next_tok]], dtype=torch.long, device=device)], dim=1)
            if next_tok == end_symbol:
                break
    return ys


def evaluate_bleu(model, dataloader, tgt_itos, device, max_len=100):
    model.eval()
    hyps, refs = [], []
    special = {PAD_IDX, SOS_IDX, EOS_IDX}
    with torch.no_grad():
        for src, tgt in dataloader:
            for i in range(src.size(0)):
                s    = src[i].unsqueeze(0).to(device)
                sm   = make_src_mask(s).to(device)
                pred = greedy_decode(model, s, sm, max_len, SOS_IDX, EOS_IDX, device)
                hyps.append(" ".join(tgt_itos[j] for j in pred[0].tolist() if j not in special))
                refs.append(" ".join(tgt_itos[j] for j in tgt[i].tolist()  if j not in special))
    return sacrebleu.corpus_bleu(hyps, [refs]).score


def run_epoch(loader, model, loss_fn, optimizer, scheduler, is_train, device):
    model.train() if is_train else model.eval()
    total_loss, total_tok = 0.0, 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for src, tgt in loader:
            src     = src.to(device)
            tgt     = tgt.to(device)
            tgt_in  = tgt[:, :-1]
            tgt_out = tgt[:, 1:]
            sm = make_src_mask(src).to(device)
            tm = make_tgt_mask(tgt_in).to(device)
            logits  = model(src, tgt_in, sm, tm)
            B, T, V = logits.shape
            loss    = loss_fn(logits.reshape(B*T, V), tgt_out.reshape(B*T))
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                if scheduler: scheduler.step()
            non_pad      = (tgt_out != PAD_IDX).sum().item()
            total_loss  += loss.item() * non_pad
            total_tok   += non_pad
    return total_loss / max(total_tok, 1)


def save_checkpoint(model, optimizer, scheduler, epoch, path="best_model.pt"):
    torch.save({
        "epoch": epoch,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "model_config": {
            "src_vocab_size": model.src_vocab_size,
            "tgt_vocab_size": model.tgt_vocab_size,
            "d_model":   model.d_model,
            "N":         len(model.encoder.layers),
            "num_heads": model.encoder.layers[0].self_attn.num_heads,
            "d_ff":      model.encoder.layers[0].ffn.linear1.out_features,
            "dropout":   0.1,
        }
    }, path)


import wandb

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

wandb.init(project="da6401-a3", name="main_training", config={
    "d_model": 256, "N": 3, "num_heads": 8, "d_ff": 512,
    "dropout": 0.1, "batch_size": 128, "num_epochs": 20,
    "warmup_steps": 4000, "smoothing": 0.1,
})

train_loader, val_loader, test_loader, vocab_info = get_dataloaders(128)
src_vocab_size = len(vocab_info["src_vocab"])
tgt_vocab_size = len(vocab_info["tgt_vocab"])
tgt_itos       = vocab_info["tgt_itos"]

model = Transformer(
    src_vocab_size=src_vocab_size, tgt_vocab_size=tgt_vocab_size,
    d_model=256, N=3, num_heads=8, d_ff=512, dropout=0.1,
    checkpoint_path=None,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = NoamScheduler(optimizer, d_model=256, warmup_steps=4000)
loss_fn   = LabelSmoothingLoss(tgt_vocab_size, PAD_IDX, 0.1)

best_val_bleu = 0.0

for epoch in range(20):
    train_loss = run_epoch(train_loader, model, loss_fn, optimizer, scheduler, True, device)
    val_loss   = run_epoch(val_loader,   model, loss_fn, None, None, False, device)
    val_bleu   = evaluate_bleu(model, val_loader, tgt_itos, device)

    print(f"Epoch {epoch+1:02d} | train={train_loss:.4f} | val={val_loss:.4f} | bleu={val_bleu:.2f}")
    wandb.log({"epoch": epoch+1, "train_loss": train_loss,
               "val_loss": val_loss, "val_bleu": val_bleu,
               "lr": optimizer.param_groups[0]["lr"]})

    if val_bleu > best_val_bleu:
        best_val_bleu = val_bleu
        save_checkpoint(model, optimizer, scheduler, epoch)
        print(f"  -> best saved (bleu={val_bleu:.2f})")

# final test bleu
test_bleu = evaluate_bleu(model, test_loader, tgt_itos, device)
print(f"\nTest BLEU: {test_bleu:.2f}")
wandb.log({"test_bleu": test_bleu})
wandb.finish()

Device: cuda


Loading train split...
Loading val split...
Loading test split...


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Epoch 01 | train=8.2738 | val=6.9563 | bleu=0.02
  -> best saved (bleu=0.02)


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


...

That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Epoch 20 | train=1.9254 | val=2.9251 | bleu=36.54


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.



Test BLEU: 37.54


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▁▁▂▂▃▃▄▄▄▅▅▆▆▇▇▇████
test_bleu,▁
train_loss,█▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁
val_bleu,▁▁▂▃▄▅▆▇▇▇██████████
val_loss,█▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,20
lr,0.00093
test_bleu,37.54189
train_loss,1.92541
val_bleu,36.54415


In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from model import Transformer, make_src_mask, make_tgt_mask
from dataset import get_dataloaders, PAD_IDX, SOS_IDX, EOS_IDX
from lr_scheduler import NoamScheduler
import sacrebleu

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size, pad_idx, smoothing=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_idx    = pad_idx
        self.smoothing  = smoothing
        self.confidence = 1.0 - smoothing
    def forward(self, logits, target):
        log_probs  = F.log_softmax(logits, dim=-1)
        smooth_val = self.smoothing / (self.vocab_size - 2)
        with torch.no_grad():
            dist = torch.full_like(log_probs, smooth_val)
            dist[:, self.pad_idx] = 0.0
            dist.scatter_(1, target.unsqueeze(1), self.confidence)
        pad_mask = (target == self.pad_idx)
        loss = -(dist * log_probs).sum(dim=-1)
        loss = loss.masked_fill(pad_mask, 0.0)
        return loss.sum() / (~pad_mask).sum().clamp(min=1)

def greedy_decode(model, src, src_mask, max_len, start_symbol, end_symbol, device):
    model.eval()
    with torch.no_grad():
        memory = model.encode(src, src_mask)
        ys = torch.tensor([[start_symbol]], dtype=torch.long, device=device)
        for _ in range(max_len - 1):
            tgt_mask = make_tgt_mask(ys).to(device)
            out      = model.decode(memory, src_mask, ys, tgt_mask)
            next_tok = out[:, -1, :].argmax(dim=-1).item()
            ys = torch.cat([ys, torch.tensor([[next_tok]], dtype=torch.long, device=device)], dim=1)
            if next_tok == end_symbol:
                break
    return ys

def evaluate_bleu(model, dataloader, tgt_itos, device, max_len=100):
    model.eval()
    hyps, refs = [], []
    special = {PAD_IDX, SOS_IDX, EOS_IDX}
    with torch.no_grad():
        for src, tgt in dataloader:
            for i in range(src.size(0)):
                s  = src[i].unsqueeze(0).to(device)
                sm = make_src_mask(s).to(device)
                pred = greedy_decode(model, s, sm, max_len, SOS_IDX, EOS_IDX, device)
                hyps.append(" ".join(tgt_itos[j] for j in pred[0].tolist() if j not in special))
                refs.append(" ".join(tgt_itos[j] for j in tgt[i].tolist()  if j not in special))
    return sacrebleu.corpus_bleu(hyps, [refs], force=True).score

def run_epoch(loader, model, loss_fn, optimizer, scheduler, is_train, device):
    model.train() if is_train else model.eval()
    total_loss, total_tok = 0.0, 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for src, tgt in loader:
            src = src.to(device); tgt = tgt.to(device)
            tgt_in  = tgt[:, :-1]; tgt_out = tgt[:, 1:]
            sm = make_src_mask(src).to(device)
            tm = make_tgt_mask(tgt_in).to(device)
            logits  = model(src, tgt_in, sm, tm)
            B, T, V = logits.shape
            loss    = loss_fn(logits.reshape(B*T, V), tgt_out.reshape(B*T))
            if is_train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                if scheduler: scheduler.step()
            non_pad      = (tgt_out != PAD_IDX).sum().item()
            total_loss  += loss.item() * non_pad
            total_tok   += non_pad
    return total_loss / max(total_tok, 1)

def build_model(src_vocab_size, tgt_vocab_size):
    return Transformer(
        src_vocab_size=src_vocab_size, tgt_vocab_size=tgt_vocab_size,
        d_model=256, N=3, num_heads=8, d_ff=512, dropout=0.1,
        checkpoint_path=None,
    ).to(DEVICE)

def train_loop(model, train_loader, val_loader, tgt_itos, tgt_vocab_size,
               num_epochs=15, fixed_lr=None, run_name="run", log_grad_norms=False):
    if fixed_lr is not None:
        optimizer = torch.optim.Adam(model.parameters(), lr=fixed_lr, betas=(0.9, 0.98), eps=1e-9)
        scheduler = None
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
        scheduler = NoamScheduler(optimizer, d_model=256, warmup_steps=4000)
    loss_fn = LabelSmoothingLoss(tgt_vocab_size, PAD_IDX, 0.1)
    for epoch in range(num_epochs):
        train_loss = run_epoch(train_loader, model, loss_fn, optimizer, scheduler, True, DEVICE)
        val_bleu   = evaluate_bleu(model, val_loader, tgt_itos, DEVICE)
        lr_now     = optimizer.param_groups[0]["lr"]
        print(f"  [{run_name}] epoch {epoch+1:02d} | loss={train_loss:.4f} | bleu={val_bleu:.2f} | lr={lr_now:.6f}")
        wandb.log({"epoch": epoch+1, "train_loss": train_loss, "val_bleu": val_bleu, "lr": lr_now})

# ── load data once, reuse for all experiments ──
print("Loading data...")
train_loader, val_loader, _, vocab_info = get_dataloaders(batch_size=128)
src_vocab_size = len(vocab_info["src_vocab"])
tgt_vocab_size = len(vocab_info["tgt_vocab"])
tgt_itos       = vocab_info["tgt_itos"]
print("Data loaded.")

# ── Experiment 1 ──
print("\n=== Experiment 1: Noam vs Fixed LR ===")
for mode in ["noam", "fixed"]:
    wandb.init(project="da6401-a3", name=f"exp1_{mode}_lr",
               group="exp1_scheduler", reinit=True)
    model = build_model(src_vocab_size, tgt_vocab_size)
    train_loop(model, train_loader, val_loader, tgt_itos, tgt_vocab_size,
               num_epochs=15, fixed_lr=(1e-4 if mode == "fixed" else None),
               run_name=f"exp1_{mode}")
    wandb.finish()
    print(f"exp1_{mode} done.")

Loading data...
Loading train split...
Loading val split...
Loading test split...
Data loaded.

=== Experiment 1: Noam vs Fixed LR ===


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


  [exp1_noam] epoch 01 | loss=8.2989 | bleu=0.01 | lr=0.000056
  [exp1_noam] epoch 02 | loss=5.9593 | bleu=2.47 | lr=0.000112
  [exp1_noam] epoch 03 | loss=4.9244 | bleu=6.01 | lr=0.000168
  [exp1_noam] epoch 04 | loss=4.4260 | bleu=10.04 | lr=0.000225
  [exp1_noam] epoch 05 | loss=3.9929 | bleu=17.77 | lr=0.000281
  [exp1_noam] epoch 06 | loss=3.6086 | bleu=22.43 | lr=0.000337
  [exp1_noam] epoch 07 | loss=3.3012 | bleu=26.75 | lr=0.000393
  [exp1_noam] epoch 08 | loss=3.0552 | bleu=30.48 | lr=0.000449
  [exp1_noam] epoch 09 | loss=2.8568 | bleu=31.51 | lr=0.000505
  [exp1_noam] epoch 10 | loss=2.6953 | bleu=32.90 | lr=0.000561
  [exp1_noam] epoch 11 | loss=2.5651 | bleu=35.28 | lr=0.000617
  [exp1_noam] epoch 12 | loss=2.4533 | bleu=35.75 | lr=0.000673
  [exp1_noam] epoch 13 | loss=2.3610 | bleu=35.56 | lr=0.000729
  [exp1_noam] epoch 14 | loss=2.2827 | bleu=35.04 | lr=0.000785
  [exp1_noam] epoch 15 | loss=2.2135 | bleu=35.66 | lr=0.000841


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▁▂▂▂▃▃▄▄▅▅▆▆▇▇█
train_loss,█▅▄▄▃▃▂▂▂▂▁▁▁▁▁
val_bleu,▁▁▂▃▄▅▆▇▇▇█████
epoch,15
lr,0.00084
train_loss,2.21352
val_bleu,35.66472


exp1_noam done.


  [exp1_fixed] epoch 01 | loss=6.5813 | bleu=2.48 | lr=0.000100
  [exp1_fixed] epoch 02 | loss=5.0398 | bleu=4.53 | lr=0.000100
  [exp1_fixed] epoch 03 | loss=4.6238 | bleu=6.79 | lr=0.000100
  [exp1_fixed] epoch 04 | loss=4.3528 | bleu=10.09 | lr=0.000100
  [exp1_fixed] epoch 05 | loss=4.1267 | bleu=12.85 | lr=0.000100
  [exp1_fixed] epoch 06 | loss=3.9291 | bleu=13.62 | lr=0.000100
  [exp1_fixed] epoch 07 | loss=3.7608 | bleu=17.45 | lr=0.000100
  [exp1_fixed] epoch 08 | loss=3.6137 | bleu=20.31 | lr=0.000100
  [exp1_fixed] epoch 09 | loss=3.4843 | bleu=21.85 | lr=0.000100
  [exp1_fixed] epoch 10 | loss=3.3675 | bleu=24.09 | lr=0.000100
  [exp1_fixed] epoch 11 | loss=3.2665 | bleu=25.70 | lr=0.000100
  [exp1_fixed] epoch 12 | loss=3.1734 | bleu=26.38 | lr=0.000100
  [exp1_fixed] epoch 13 | loss=3.0878 | bleu=26.39 | lr=0.000100
  [exp1_fixed] epoch 14 | loss=3.0120 | bleu=28.52 | lr=0.000100
  [exp1_fixed] epoch 15 | loss=2.9398 | bleu=29.02 | lr=0.000100


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▅▄▄▃▃▃▂▂▂▂▁▁▁▁
val_bleu,▁▂▂▃▄▄▅▆▆▇▇▇▇██
epoch,15
lr,0.0001
train_loss,2.93983
val_bleu,29.01702


exp1_fixed done.


In [17]:
# ── Experiment 2: With vs Without 1/sqrt(dk) scaling ──

class UnscaledMHA(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, S, _ = x.size()
        return x.view(B, S, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, q, k, v, mask=None):
        B  = q.size(0)
        Q  = self.split_heads(self.W_q(q))
        K  = self.split_heads(self.W_k(k))
        V  = self.split_heads(self.W_v(v))
        scores = torch.matmul(Q, K.transpose(-2, -1))  # no 1/sqrt(dk)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        attn_w = torch.nan_to_num(F.softmax(scores, dim=-1), nan=0.0)
        out    = torch.matmul(attn_w, V).transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.W_o(out)

print("\n=== Experiment 2: Scaling Factor ===")
for use_scale in [True, False]:
    name = "with_scale" if use_scale else "no_scale"
    wandb.init(project="da6401-a3", name=f"exp2_{name}",
               group="exp2_scaling", reinit=True)

    model = build_model(src_vocab_size, tgt_vocab_size)

    if not use_scale:
        for enc_layer in model.encoder.layers:
            enc_layer.self_attn = UnscaledMHA(256, 8).to(DEVICE)
        for dec_layer in model.decoder.layers:
            dec_layer.self_attn  = UnscaledMHA(256, 8).to(DEVICE)
            dec_layer.cross_attn = UnscaledMHA(256, 8).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
    scheduler = NoamScheduler(optimizer, d_model=256, warmup_steps=4000)
    loss_fn   = LabelSmoothingLoss(tgt_vocab_size, PAD_IDX, 0.1)

    for epoch in range(10):
        model.train()
        total_loss, total_tok = 0.0, 0
        grad_norms_q, grad_norms_k = [], []

        for src, tgt in train_loader:
            src = src.to(DEVICE); tgt = tgt.to(DEVICE)
            tgt_in  = tgt[:, :-1]; tgt_out = tgt[:, 1:]
            sm = make_src_mask(src).to(DEVICE)
            tm = make_tgt_mask(tgt_in).to(DEVICE)
            logits  = model(src, tgt_in, sm, tm)
            B, T, V = logits.shape
            loss    = loss_fn(logits.reshape(B*T, V), tgt_out.reshape(B*T))
            optimizer.zero_grad()
            loss.backward()

            # collect Q and K grad norms
            for n, p in model.named_parameters():
                if p.grad is not None:
                    if 'W_q' in n: grad_norms_q.append(p.grad.norm().item())
                    if 'W_k' in n: grad_norms_k.append(p.grad.norm().item())

            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            non_pad      = (tgt_out != PAD_IDX).sum().item()
            total_loss  += loss.item() * non_pad
            total_tok   += non_pad

        train_loss  = total_loss / max(total_tok, 1)
        val_bleu    = evaluate_bleu(model, val_loader, tgt_itos, DEVICE)
        avg_q_norm  = sum(grad_norms_q) / max(len(grad_norms_q), 1)
        avg_k_norm  = sum(grad_norms_k) / max(len(grad_norms_k), 1)

        print(f"  [{name}] epoch {epoch+1:02d} | loss={train_loss:.4f} | bleu={val_bleu:.2f} | grad_Q={avg_q_norm:.4f} | grad_K={avg_k_norm:.4f}")
        wandb.log({
            "epoch":      epoch + 1,
            "train_loss": train_loss,
            "val_bleu":   val_bleu,
            "avg_grad_norm_Q": avg_q_norm,
            "avg_grad_norm_K": avg_k_norm,
        })

    wandb.finish()
    print(f"exp2_{name} done.")


=== Experiment 2: Scaling Factor ===


  [with_scale] epoch 01 | loss=8.2730 | bleu=0.02 | grad_Q=0.0017 | grad_K=0.0016
  [with_scale] epoch 02 | loss=5.9405 | bleu=2.50 | grad_Q=0.0050 | grad_K=0.0038
  [with_scale] epoch 03 | loss=4.9210 | bleu=6.13 | grad_Q=0.0145 | grad_K=0.0113
  [with_scale] epoch 04 | loss=4.4124 | bleu=11.76 | grad_Q=0.0207 | grad_K=0.0163
  [with_scale] epoch 05 | loss=3.9794 | bleu=16.39 | grad_Q=0.0243 | grad_K=0.0202
  [with_scale] epoch 06 | loss=3.6100 | bleu=21.82 | grad_Q=0.0278 | grad_K=0.0237
  [with_scale] epoch 07 | loss=3.3062 | bleu=27.02 | grad_Q=0.0299 | grad_K=0.0259
  [with_scale] epoch 08 | loss=3.0628 | bleu=28.98 | grad_Q=0.0316 | grad_K=0.0277
  [with_scale] epoch 09 | loss=2.8635 | bleu=31.34 | grad_Q=0.0326 | grad_K=0.0288
  [with_scale] epoch 10 | loss=2.7026 | bleu=33.78 | grad_Q=0.0323 | grad_K=0.0291


avg_grad_norm_K,▁▂▃▅▆▇▇███
avg_grad_norm_Q,▁▂▄▅▆▇▇███
epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▅▄▃▃▂▂▁▁▁
val_bleu,▁▂▂▃▄▆▇▇▇█
avg_grad_norm_K,0.02906
avg_grad_norm_Q,0.03228
epoch,10
train_loss,2.70259
val_bleu,33.78346


exp2_with_scale done.


  [no_scale] epoch 01 | loss=8.2534 | bleu=0.02 | grad_Q=0.0039 | grad_K=0.0037
  [no_scale] epoch 02 | loss=5.8612 | bleu=2.47 | grad_Q=0.0207 | grad_K=0.0146
  [no_scale] epoch 03 | loss=4.8124 | bleu=7.39 | grad_Q=0.0398 | grad_K=0.0287
  [no_scale] epoch 04 | loss=4.2947 | bleu=12.37 | grad_Q=0.0474 | grad_K=0.0375
  [no_scale] epoch 05 | loss=3.8997 | bleu=18.33 | grad_Q=0.0507 | grad_K=0.0423
  [no_scale] epoch 06 | loss=3.5511 | bleu=23.80 | grad_Q=0.0564 | grad_K=0.0483
  [no_scale] epoch 07 | loss=3.2587 | bleu=27.12 | grad_Q=0.0598 | grad_K=0.0524
  [no_scale] epoch 08 | loss=3.0173 | bleu=31.48 | grad_Q=0.0610 | grad_K=0.0545
  [no_scale] epoch 09 | loss=2.8218 | bleu=33.33 | grad_Q=0.0621 | grad_K=0.0561
  [no_scale] epoch 10 | loss=2.6668 | bleu=34.63 | grad_Q=0.0613 | grad_K=0.0560


avg_grad_norm_K,▁▂▄▆▆▇████
avg_grad_norm_Q,▁▃▅▆▇▇████
epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▅▄▃▃▂▂▁▁▁
val_bleu,▁▁▂▃▅▆▆▇██
avg_grad_norm_K,0.05599
avg_grad_norm_Q,0.06135
epoch,10
train_loss,2.66677
val_bleu,34.63204


exp2_no_scale done.


In [19]:
# ── Experiment 3: Attention Head Visualisation ──
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

print("\n=== Experiment 3: Attention Heatmaps ===")
wandb.init(project="da6401-a3", name="exp3_attn_heads",
           group="exp3_heads", reinit=True)

model3    = build_model(src_vocab_size, tgt_vocab_size)
optimizer3 = torch.optim.Adam(model3.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler3 = NoamScheduler(optimizer3, d_model=256, warmup_steps=4000)
loss_fn3   = LabelSmoothingLoss(tgt_vocab_size, PAD_IDX, 0.1)

# train for 15 epochs
for epoch in range(15):
    train_loss = run_epoch(train_loader, model3, loss_fn3, optimizer3, scheduler3, True, DEVICE)
    val_bleu   = evaluate_bleu(model3, val_loader, tgt_itos, DEVICE)
    print(f"  [exp3] epoch {epoch+1:02d} | loss={train_loss:.4f} | bleu={val_bleu:.2f}")
    wandb.log({"epoch": epoch+1, "train_loss": train_loss, "val_bleu": val_bleu})

# extract attention weights from last encoder layer
model3.eval()
src_itos = vocab_info["src_itos"]
src_batch, _ = next(iter(val_loader))
src_single   = src_batch[:1].to(DEVICE)

src_tokens = [src_itos.get(i.item(), "<unk>") for i in src_single[0]
              if i.item() not in {PAD_IDX, SOS_IDX, EOS_IDX}]

attn_store = {}
def hook_fn(module, inp, out):
    q, k = inp[0], inp[1]
    Q = module.split_heads(module.W_q(q))
    K = module.split_heads(module.W_k(k))
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(module.d_k)
    attn_store['w'] = F.softmax(scores, dim=-1).detach().cpu()

handle = model3.encoder.layers[-1].self_attn.register_forward_hook(hook_fn)
with torch.no_grad():
    model3.encode(src_single, make_src_mask(src_single).to(DEVICE))
handle.remove()

attn_w    = attn_store['w'][0]   # [num_heads, seq, seq]
num_heads = attn_w.shape[0]
seq_len   = len(src_tokens)

fig, axes = plt.subplots(2, num_heads // 2, figsize=(20, 8))
for h, ax in enumerate(axes.flatten()):
    data = attn_w[h, :seq_len, :seq_len].numpy()
    ax.imshow(data, cmap='Blues')
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(src_tokens, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(src_tokens, fontsize=7)
    ax.set_title(f"Head {h+1}", fontsize=9)

plt.suptitle("Last Encoder Layer — All Attention Heads")
plt.tight_layout()
plt.savefig("attention_heads.png", dpi=100)
wandb.log({"attention_heatmaps": wandb.Image("attention_heads.png")})
print("Heatmaps logged to W&B.")
wandb.finish()
print("exp3 done.")


=== Experiment 3: Attention Heatmaps ===


  [exp3] epoch 01 | loss=8.2732 | bleu=0.02
  [exp3] epoch 02 | loss=5.9348 | bleu=2.50
  [exp3] epoch 03 | loss=4.9078 | bleu=5.86
  [exp3] epoch 04 | loss=4.4087 | bleu=11.17
  [exp3] epoch 05 | loss=3.9756 | bleu=18.08
  [exp3] epoch 06 | loss=3.6045 | bleu=22.90
  [exp3] epoch 07 | loss=3.2981 | bleu=26.82
  [exp3] epoch 08 | loss=3.0530 | bleu=29.09
  [exp3] epoch 09 | loss=2.8548 | bleu=32.06
  [exp3] epoch 10 | loss=2.6929 | bleu=34.31
  [exp3] epoch 11 | loss=2.5594 | bleu=34.58
  [exp3] epoch 12 | loss=2.4502 | bleu=34.73
  [exp3] epoch 13 | loss=2.3560 | bleu=36.53
  [exp3] epoch 14 | loss=2.2788 | bleu=36.69
  [exp3] epoch 15 | loss=2.2081 | bleu=36.27
Heatmaps logged to W&B.


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▅▄▄▃▃▂▂▂▂▁▁▁▁▁
val_bleu,▁▁▂▃▄▅▆▇▇██████
epoch,15
train_loss,2.20811
val_bleu,36.27117


exp3 done.


In [ ]:
# ── Experiment 4: Sinusoidal vs Learned Positional Encoding ──

class LearnedPE(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=256):
        super().__init__()
        self.dropout   = nn.Dropout(p=dropout)
        self.pos_embed = nn.Embedding(max_len, d_model)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        return self.dropout(x + self.pos_embed(positions))

print("\n=== Experiment 4: Sinusoidal vs Learned PE ===")
for pe_type in ["sinusoidal", "learned"]:
    wandb.init(project="da6401-a3", name=f"exp4_{pe_type}",
               group="exp4_pos_enc", reinit=True)

    model4 = build_model(src_vocab_size, tgt_vocab_size)
    if pe_type == "learned":
        model4.pos_enc = LearnedPE(256, max_len=256).to(DEVICE)

    optimizer4 = torch.optim.Adam(model4.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
    scheduler4 = NoamScheduler(optimizer4, d_model=256, warmup_steps=4000)
    loss_fn4   = LabelSmoothingLoss(tgt_vocab_size, PAD_IDX, 0.1)

    for epoch in range(15):
        train_loss = run_epoch(train_loader, model4, loss_fn4, optimizer4, scheduler4, True, DEVICE)
        val_bleu   = evaluate_bleu(model4, val_loader, tgt_itos, DEVICE)
        print(f"  [exp4_{pe_type}] epoch {epoch+1:02d} | loss={train_loss:.4f} | bleu={val_bleu:.2f}")
        wandb.log({"epoch": epoch+1, "train_loss": train_loss, "val_bleu": val_bleu})

    wandb.finish()
    print(f"exp4_{pe_type} done.")


=== Experiment 4: Sinusoidal vs Learned PE ===


  [exp4_sinusoidal] epoch 01 | loss=8.2862 | bleu=0.02
  [exp4_sinusoidal] epoch 02 | loss=5.9349 | bleu=2.42
  [exp4_sinusoidal] epoch 03 | loss=4.9220 | bleu=4.98
  [exp4_sinusoidal] epoch 04 | loss=4.4414 | bleu=10.02
  [exp4_sinusoidal] epoch 05 | loss=4.0161 | bleu=16.84
  [exp4_sinusoidal] epoch 06 | loss=3.6329 | bleu=22.46
  [exp4_sinusoidal] epoch 07 | loss=3.3210 | bleu=25.34
  [exp4_sinusoidal] epoch 08 | loss=3.0699 | bleu=29.37
  [exp4_sinusoidal] epoch 09 | loss=2.8666 | bleu=31.78
  [exp4_sinusoidal] epoch 10 | loss=2.7018 | bleu=33.89
  [exp4_sinusoidal] epoch 11 | loss=2.5667 | bleu=35.65
  [exp4_sinusoidal] epoch 12 | loss=2.4581 | bleu=35.76
  [exp4_sinusoidal] epoch 13 | loss=2.3613 | bleu=36.19
  [exp4_sinusoidal] epoch 14 | loss=2.2828 | bleu=36.71
  [exp4_sinusoidal] epoch 15 | loss=2.2136 | bleu=36.18


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▅▄▄▃▃▂▂▂▂▁▁▁▁▁
val_bleu,▁▁▂▃▄▅▆▇▇▇█████
epoch,15
train_loss,2.2136
val_bleu,36.18435


exp4_sinusoidal done.


  [exp4_learned] epoch 01 | loss=8.3262 | bleu=0.02
  [exp4_learned] epoch 02 | loss=5.9510 | bleu=2.47
  [exp4_learned] epoch 03 | loss=5.0012 | bleu=4.46
  [exp4_learned] epoch 04 | loss=4.5465 | bleu=7.45
  [exp4_learned] epoch 05 | loss=4.1640 | bleu=14.28
  [exp4_learned] epoch 06 | loss=3.8213 | bleu=19.55


# session restart

In [2]:
!pip install wandb sacrebleu gdown datasets spacy -q
!python -m spacy download en_core_web_sm -q
!python -m spacy download de_core_news_sm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 53.7 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 73.1 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: da25m017 (da25m017-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import torch, torch.nn as nn, torch.nn.functional as F, math, sacrebleu
from model import Transformer, make_src_mask, make_tgt_mask
from dataset import get_dataloaders, PAD_IDX, SOS_IDX, EOS_IDX
from lr_scheduler import NoamScheduler
import wandb

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# paste the LabelSmoothingLoss, greedy_decode, evaluate_bleu, run_epoch, build_model, train_loop helper functions here
# (same ones from before)

print("Loading data...")
train_loader, val_loader, _, vocab_info = get_dataloaders(batch_size=128)
src_vocab_size = len(vocab_info["src_vocab"])
tgt_vocab_size = len(vocab_info["tgt_vocab"])
tgt_itos       = vocab_info["tgt_itos"]
print("Data loaded.")

ModuleNotFoundError: No module named 'model'

In [15]:
# ── Experiment 5: Label Smoothing eps=0.1 vs eps=0.0 ──

print("\n=== Experiment 5: Label Smoothing ===")
for eps in [0.1, 0.0]:
    name = f"eps_{str(eps).replace('.','')}"
    wandb.init(project="da6401-a3", name=f"exp5_{name}",
               group="exp5_smoothing", reinit=True)

    model5     = build_model(src_vocab_size, tgt_vocab_size)
    optimizer5 = torch.optim.Adam(model5.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
    scheduler5 = NoamScheduler(optimizer5, d_model=256, warmup_steps=4000)
    loss_fn5   = LabelSmoothingLoss(tgt_vocab_size, PAD_IDX, smoothing=eps)

    for epoch in range(15):
        model5.train()
        total_loss, total_tok = 0.0, 0
        conf_sum, conf_count  = 0.0, 0

        for src, tgt in train_loader:
            src     = src.to(DEVICE)
            tgt     = tgt.to(DEVICE)
            tgt_in  = tgt[:, :-1]
            tgt_out = tgt[:, 1:]
            sm      = make_src_mask(src).to(DEVICE)
            tm      = make_tgt_mask(tgt_in).to(DEVICE)
            logits  = model5(src, tgt_in, sm, tm)
            B, T, V = logits.shape
            flat_l  = logits.reshape(B*T, V)
            flat_t  = tgt_out.reshape(B*T)
            loss    = loss_fn5(flat_l, flat_t)

            optimizer5.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model5.parameters(), 1.0)
            optimizer5.step()
            scheduler5.step()

            with torch.no_grad():
                probs   = F.softmax(flat_l, dim=-1)
                non_pad = flat_t != PAD_IDX
                conf    = probs[non_pad].gather(1, flat_t[non_pad].unsqueeze(1)).mean()
                conf_sum   += conf.item()
                conf_count += 1

            non_pad_n   = (tgt_out != PAD_IDX).sum().item()
            total_loss += loss.item() * non_pad_n
            total_tok  += non_pad_n

        train_loss = total_loss / max(total_tok, 1)
        val_bleu   = evaluate_bleu(model5, val_loader, tgt_itos, DEVICE)
        avg_conf   = conf_sum / max(conf_count, 1)

        print(f"  [eps={eps}] epoch {epoch+1:02d} | loss={train_loss:.4f} | bleu={val_bleu:.2f} | conf={avg_conf:.4f}")
        wandb.log({"epoch": epoch+1, "train_loss": train_loss,
                   "val_bleu": val_bleu, "prediction_confidence": avg_conf})

    wandb.finish()
    print(f"exp5_{name} done.")

print("\nAll experiments complete!")


=== Experiment 5: Label Smoothing ===


  [eps=0.1] epoch 01 | loss=8.2688 | bleu=0.02 | conf=0.0007
  [eps=0.1] epoch 02 | loss=5.9406 | bleu=2.53 | conf=0.0513
  [eps=0.1] epoch 03 | loss=4.9192 | bleu=6.33 | conf=0.1785
  [eps=0.1] epoch 04 | loss=4.4157 | bleu=10.56 | conf=0.2455
  [eps=0.1] epoch 05 | loss=3.9839 | bleu=16.31 | conf=0.3058
  [eps=0.1] epoch 06 | loss=3.6111 | bleu=23.43 | conf=0.3589
  [eps=0.1] epoch 07 | loss=3.3047 | bleu=26.26 | conf=0.4036
  [eps=0.1] epoch 08 | loss=3.0518 | bleu=30.86 | conf=0.4421
  [eps=0.1] epoch 09 | loss=2.8533 | bleu=31.76 | conf=0.4741
  [eps=0.1] epoch 10 | loss=2.6921 | bleu=33.20 | conf=0.5013
  [eps=0.1] epoch 11 | loss=2.5593 | bleu=34.18 | conf=0.5250
  [eps=0.1] epoch 12 | loss=2.4500 | bleu=35.38 | conf=0.5451
  [eps=0.1] epoch 13 | loss=2.3598 | bleu=35.80 | conf=0.5625
  [eps=0.1] epoch 14 | loss=2.2781 | bleu=35.72 | conf=0.5788
  [eps=0.1] epoch 15 | loss=2.2107 | bleu=37.22 | conf=0.5930


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
prediction_confidence,▁▂▃▄▅▅▆▆▇▇▇▇███
train_loss,█▅▄▄▃▃▂▂▂▂▁▁▁▁▁
val_bleu,▁▁▂▃▄▅▆▇▇▇▇████
epoch,15
prediction_confidence,0.593
train_loss,2.21073
val_bleu,37.22297


exp5_eps_01 done.


  [eps=0.0] epoch 01 | loss=8.1635 | bleu=0.02 | conf=0.0008
  [eps=0.0] epoch 02 | loss=5.4750 | bleu=2.48 | conf=0.0522
  [eps=0.0] epoch 03 | loss=4.2491 | bleu=5.74 | conf=0.1936
  [eps=0.0] epoch 04 | loss=3.6691 | bleu=10.33 | conf=0.2673
  [eps=0.0] epoch 05 | loss=3.1630 | bleu=16.71 | conf=0.3346
  [eps=0.0] epoch 06 | loss=2.7158 | bleu=21.45 | conf=0.3939
  [eps=0.0] epoch 07 | loss=2.3500 | bleu=26.10 | conf=0.4441
  [eps=0.0] epoch 08 | loss=2.0514 | bleu=29.75 | conf=0.4861
  [eps=0.0] epoch 09 | loss=1.8038 | bleu=31.67 | conf=0.5226
  [eps=0.0] epoch 10 | loss=1.6000 | bleu=32.88 | conf=0.5542
  [eps=0.0] epoch 11 | loss=1.4314 | bleu=35.00 | conf=0.5812
  [eps=0.0] epoch 12 | loss=1.2884 | bleu=35.53 | conf=0.6047
  [eps=0.0] epoch 13 | loss=1.1648 | bleu=35.58 | conf=0.6265
  [eps=0.0] epoch 14 | loss=1.0561 | bleu=36.55 | conf=0.6462
  [eps=0.0] epoch 15 | loss=0.9644 | bleu=36.31 | conf=0.6646


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
prediction_confidence,▁▂▃▄▅▅▆▆▇▇▇▇███
train_loss,█▅▄▄▃▃▂▂▂▂▁▁▁▁▁
val_bleu,▁▁▂▃▄▅▆▇▇▇█████
epoch,15
prediction_confidence,0.66456
train_loss,0.96441
val_bleu,36.31456


exp5_eps_00 done.

All experiments complete!


In [13]:
model_code = open("/kaggle/working/model.py").read()
print(model_code[4800:5100])  # check what's actually in the file around line 143

d(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

class Transformer(nn.Module):
    def __init__(self, src_vocab_size=18669, tgt_vocab_size=9797,
                 d_model=256, N=3, num_heads=8, 


In [14]:
import sys

# remove all cached modules
for mod in ['model', 'dataset', 'lr_scheduler']:
    if mod in sys.modules:
        del sys.modules[mod]

# reimport fresh
from model import Transformer, make_src_mask, make_tgt_mask
from dataset import get_dataloaders, PAD_IDX, SOS_IDX, EOS_IDX
from lr_scheduler import NoamScheduler

# redefine build_model
def build_model(src_vocab_size, tgt_vocab_size):
    return Transformer(
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        d_model=256, N=3, num_heads=8, d_ff=512, dropout=0.1,
        checkpoint_path=None
    ).to(DEVICE)

print("Modules reloaded. Testing build_model...")
m = build_model(src_vocab_size, tgt_vocab_size)
print("build_model OK:", type(m))
del m

Modules reloaded. Testing build_model...
build_model OK: <class 'model.Transformer'>
